In [4]:
import numpy as np
import torch
from transformers import pipeline

c:\Users\sange\Desktop\emotional_int_assessment\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

inputs = tokenizer("I would feel frustrated initially, but I understand people can make mistakes.", return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
print(logits)
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]


tensor([[-0.8473,  0.8732]])


'POSITIVE'

In [5]:
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

emotion_analyzer = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=True
)


Device set to use cpu
Device set to use cpu
c:\Users\sange\Desktop\emotional_int_assessment\.venv\lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [6]:
user_profile = {
    "age": 24,
    "gender": "Female",
    "profession": "Software Engineer"
}


In [7]:
scenario = (
    "You are working on a critical project with a tight deadline. "
    "A teammate misses an important task, affecting progress."
)

questions = [
    "How would you feel in this situation?",
    "How would you respond to your teammate?",
    "What steps would you take to manage your emotions?"
]

print("Scenario:\n", scenario)
print("\nQuestions:")
for q in questions:
    print("-", q)


Scenario:
 You are working on a critical project with a tight deadline. A teammate misses an important task, affecting progress.

Questions:
- How would you feel in this situation?
- How would you respond to your teammate?
- What steps would you take to manage your emotions?


In [8]:
responses = [
    "I would feel frustrated initially, but I understand people can make mistakes.",
    "I would talk calmly with them and try to find a solution together.",
    "I would take a moment to calm myself and focus on resolving the issue."
]


In [9]:
def validate_response(text, min_words=5):
    if len(text.split()) < min_words:
        return False
    return True

validated_responses = [r for r in responses if validate_response(r)]

validated_responses


['I would feel frustrated initially, but I understand people can make mistakes.',
 'I would talk calmly with them and try to find a solution together.',
 'I would take a moment to calm myself and focus on resolving the issue.']

In [21]:
def get_sentiment_scores(text):
    result = sentiment_analyzer(text)[0]
    return result["label"], result["score"]

sentiment_results = [get_sentiment_scores(r) for r in validated_responses]
sentiment_results


[('POSITIVE', 0.8481934070587158),
 ('NEGATIVE', 0.9950717091560364),
 ('NEGATIVE', 0.9970566034317017)]

In [11]:
def get_emotion_scores(text):
    emotions = emotion_analyzer(text)[0]
    return {e["label"]: e["score"] for e in emotions}

emotion_results = [get_emotion_scores(r) for r in validated_responses]
emotion_results


[{'anger': 0.987751841545105,
  'disgust': 0.00292025925591588,
  'fear': 0.0013783269096165895,
  'joy': 0.0004166270955465734,
  'neutral': 0.002982943784445524,
  'sadness': 0.003522442886605859,
  'surprise': 0.0010274965316057205},
 {'anger': 0.37216252088546753,
  'disgust': 0.06587148457765579,
  'fear': 0.043937575072050095,
  'joy': 0.08825377374887466,
  'neutral': 0.39161422848701477,
  'sadness': 0.03613114729523659,
  'surprise': 0.002029233844950795},
 {'anger': 0.02386077493429184,
  'disgust': 0.025029970332980156,
  'fear': 0.028145598247647285,
  'joy': 0.07496796548366547,
  'neutral': 0.6446672081947327,
  'sadness': 0.2000001072883606,
  'surprise': 0.0033283112570643425}]

In [12]:
EQ_CATEGORIES = {
    "self_awareness": ["joy", "sadness"],
    "emotional_resilience": ["fear", "sadness"],
    "conflict_resolution": ["anger"],
    "empathy": ["joy"],
    "emotional_regulation": ["anger", "fear"]
}


In [13]:
def calculate_eq_scores(emotion_results):
    scores = {cat: 0.0 for cat in EQ_CATEGORIES}

    for emotions in emotion_results:
        for category, relevant_emotions in EQ_CATEGORIES.items():
            for emo in relevant_emotions:
                scores[category] += emotions.get(emo, 0)

    # Normalize
    for k in scores:
        scores[k] = round(scores[k] / len(emotion_results), 3)

    return scores

eq_scores = calculate_eq_scores(emotion_results)
eq_scores


{'self_awareness': 0.134,
 'emotional_resilience': 0.104,
 'conflict_resolution': 0.461,
 'empathy': 0.055,
 'emotional_regulation': 0.486}

In [14]:
overall_eq = round(np.mean(list(eq_scores.values())) * 100, 2)
overall_eq


np.float64(24.8)

In [15]:
def interpret_eq(score):
    if score < 40:
        return "Low EQ – Needs emotional skill development"
    elif score < 70:
        return "Average EQ – Good emotional awareness"
    else:
        return "High EQ – Strong emotional intelligence"

interpretation = interpret_eq(overall_eq)
interpretation


'Low EQ – Needs emotional skill development'

In [16]:
print("User Profile:", user_profile)
print("Overall EQ Score:", overall_eq)
print("Interpretation:", interpretation)
print("\nCategory Scores:")
for k, v in eq_scores.items():
    print(f"{k}: {v}")


User Profile: {'age': 24, 'gender': 'Female', 'profession': 'Software Engineer'}
Overall EQ Score: 24.8
Interpretation: Low EQ – Needs emotional skill development

Category Scores:
self_awareness: 0.134
emotional_resilience: 0.104
conflict_resolution: 0.461
empathy: 0.055
emotional_regulation: 0.486
